# Cohort Retention Analysis
**Project:** Patreon Subscriber Analytics  
**Author:** Eduard-Daniel Acsente  
**Last updated:** 2026-04-23

## What this notebook does
Starting from a cleaned, anonymized patron export, we build a cohort retention matrix —
the same analysis used by SaaS companies to understand how well they keep subscribers over time.

## Questions we want to answer
1. What % of patrons from each signup month are still subscribed after 1, 3, 6, 12 months?
2. Which cohort retains best? Which retains worst?
3. At what month does the biggest drop-off typically happen?
4. Are newer cohorts improving or declining compared to older ones?

---

## 0. Imports
We only need three libraries for this entire notebook.
- `pandas` — data manipulation
- `numpy` — math helpers (floor, etc.)
- `matplotlib` — plotting the heatmap

**Your task:** Run this cell. If any library is missing, install it with `pip install <name>` in your terminal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# This just makes plots look cleaner
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'monospace'

print('Libraries loaded OK')

---
## 1. Load the data

We load `patrons_clean.csv` — the anonymized file you produced in the anonymization step.

**Your task:** Run this cell and look at the output.
- How many rows are there?
- What does each row represent?
- What do the date columns look like?

In [ ]:
df = pd.read_csv(
    '../data/processed/patrons_clean.csv',
    parse_dates=['started_at', 'last_updated_at', 'last_charge_at']
)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

---
## 2. Filter to paying patrons only

Free members never paid anything, so they can't "churn" in a meaningful revenue sense.
We only want patrons where `pledge_usd > 0`.

**Your task:** Fill in the filter condition below, then run the cell.
How many paying patrons do you have? How many were active vs former?

In [ ]:
# YOUR CODE: filter df to only rows where pledge_usd > 0
paying = df[___]

print(f'Total paying patrons: {len(paying)}')
print()
print('Patron status breakdown:')
print(paying['patron_status'].value_counts())
print()
print('Tier breakdown:')
print(paying['tier'].value_counts())

---
## 3. Determine each patron's end date

To calculate how long someone stayed, we need two dates:
- **Start date:** `started_at` — when they first subscribed
- **End date:** depends on their status
  - If **active** → they're still here, so end date = today
  - If **former/declined** → they left, so end date ≈ `last_updated_at`

This is a common pattern in subscription analytics called **"right-censoring"** —
active subscribers haven't churned yet, so we use today as their provisional end date.

**Your task:** Read the code below carefully. Make sure you understand *why* we use
`last_updated_at` for former patrons and not `last_charge_at`. (Hint: think about what
each column actually represents.)

In [ ]:
TODAY = pd.Timestamp('2026-04-23')

paying = paying.copy()  # avoid modifying the original dataframe

paying['end_date'] = paying.apply(
    lambda row: TODAY if row['is_active'] else row['last_updated_at'],
    axis=1
)

# Sanity check: active patrons should all have end_date = TODAY
print('Active patrons end_date (should all be 2026-04-23):')
print(paying[paying['is_active']]['end_date'].value_counts().head())

# Sanity check: former patrons should have varied end dates
print('\nFormer patrons end_date sample:')
print(paying[~paying['is_active']]['end_date'].sort_values().tail(10).values)

---
## 4. Calculate tenure in months

Tenure = how many complete months a patron was subscribed.

Formula: `floor((end_date - started_at).days / 30.44)`

We use 30.44 because that's the average number of days in a month (365 / 12).
We use `floor()` (round down) because a patron needs to complete a full month to count.

**Your task:** Fill in the formula below.

In [ ]:
# YOUR CODE: calculate tenure in months
# Hint: (paying['end_date'] - paying['started_at']).dt.days gives you days
paying['tenure_months'] = ___

print('Tenure stats (months):')
print(paying['tenure_months'].describe())
print()
print('Distribution:')
print(paying['tenure_months'].value_counts().sort_index().head(20))

---
## 5. Assign each patron to a cohort

A **cohort** is a group of patrons who joined in the same calendar month.
For example, everyone who subscribed in November 2024 is the "2024-11" cohort.

In pandas, `dt.to_period('M')` converts a datetime to a year-month period.

**Your task:** Run this cell and look at the cohort sizes.
Which month had the most new paying patrons?

In [ ]:
paying['cohort'] = paying['started_at'].dt.to_period('M')

cohort_sizes = paying.groupby('cohort')['patron_id'].count().rename('size')

print('Patrons per cohort:')
print(cohort_sizes.to_string())
print(f'\nTotal cohorts: {len(cohort_sizes)}')
print(f'Largest cohort: {cohort_sizes.idxmax()} ({cohort_sizes.max()} patrons)')

---
## 6. Build the retention matrix

This is the core of the analysis. For each cohort and each month M, we want to know:
> "What % of patrons from this cohort were still subscribed at month M?"

A patron counts as "retained at month M" if their `tenure_months >= M`.

The logic:
1. Loop over each cohort
2. For each month 0 through 12:
   - Count how many patrons in that cohort have `tenure_months >= month`
   - Divide by cohort size → retention rate
3. Skip months the cohort hasn't reached yet (would give false 0%)

**Your task:** Read through this code carefully.
The key line is the one with `>= month` — make sure you understand why that condition
gives us retention and not churn.

In [ ]:
MAX_MONTHS = 12

retention_rows = []

for cohort, group in paying.groupby('cohort'):
    size = len(group)
    cohort_age_months = (TODAY - cohort.to_timestamp()).days / 30.44

    for month in range(MAX_MONTHS + 1):

        # Don't calculate retention for months the cohort hasn't reached yet
        if cohort_age_months < month:
            continue

        survived = (group['tenure_months'] >= month).sum()
        retention_pct = round(survived / size * 100, 1)

        retention_rows.append({
            'cohort': str(cohort),
            'month': month,
            'retention_pct': retention_pct,
            'patrons_retained': int(survived),
            'cohort_size': size,
        })

retention = pd.DataFrame(retention_rows)

print(f'Retention table shape: {retention.shape}')
print()
print('First 15 rows:')
print(retention.head(15).to_string(index=False))

---
## 7. Pivot into a matrix

Right now retention is a long table (one row per cohort+month combination).
For a heatmap we need a wide table: cohorts as rows, months as columns.

`pd.pivot_table()` does exactly this transformation.

**Your task:** Run this and look at the shape. Can you spot which cohort has
the best 12-month retention with your eyes?

In [ ]:
matrix = retention.pivot_table(
    index='cohort',
    columns='month',
    values='retention_pct'
)

# Rename columns for readability
matrix.columns = [f'M{m}' for m in matrix.columns]

print(f'Matrix shape: {matrix.shape}  (cohorts x months)')
print()
matrix

---
## 8. Visualize: the heatmap

Now we turn the matrix into a color-coded heatmap.
Green = high retention, Red = low retention.

**Your task:** Run this cell to see your first chart.
Then experiment:
- Change `cmap='RdYlGn'` to `cmap='Blues'` — what happens?
- Change `vmin=0` to `vmin=50` — how does the color scale change?
- Try adding `annot_kws={'size': 7}` inside `plt.table` — what does it do?

Breaking things on purpose is how you learn what each parameter does.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))

# Add cohort sizes as a label column
cohort_size_map = paying.groupby(paying['cohort'].astype(str))['patron_id'].count()
matrix_labeled = matrix.copy()

# Color map: red → yellow → green
cmap = plt.cm.RdYlGn
norm = mcolors.Normalize(vmin=0, vmax=100)

im = ax.imshow(
    matrix_labeled.values,
    aspect='auto',
    cmap=cmap,
    norm=norm,
)

# Add text labels inside each cell
for i in range(matrix_labeled.shape[0]):
    for j in range(matrix_labeled.shape[1]):
        val = matrix_labeled.iloc[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.0f}%', ha='center', va='center',
                    fontsize=7, color='black' if 30 < val < 80 else 'white',
                    fontweight='bold')

# Axis labels
ax.set_xticks(range(len(matrix_labeled.columns)))
ax.set_xticklabels(matrix_labeled.columns, fontsize=9)
ax.set_yticks(range(len(matrix_labeled.index)))
ax.set_yticklabels(
    [f"{c}  (n={cohort_size_map.get(c, '?')})" for c in matrix_labeled.index],
    fontsize=8
)

ax.set_title('Patron Cohort Retention — Paying Members Only', fontsize=14, pad=16, fontweight='bold')
ax.set_xlabel('Months Since First Payment', fontsize=10)
ax.set_ylabel('Cohort (Signup Month)', fontsize=10)

plt.colorbar(im, ax=ax, label='Retention %', shrink=0.6)
plt.tight_layout()
plt.savefig('outputs/cohort_retention_heatmap.png', bbox_inches='tight')
plt.show()
print('Chart saved to outputs/cohort_retention_heatmap.png')

---
## 9. Average retention curve

The heatmap shows individual cohorts. Now let's compute the *average* retention
across all cohorts at each month — this gives us a single "how does the typical
patron behave" curve.

**Your task:** Fill in the groupby below.
What month shows the biggest average drop?

In [ ]:
# YOUR CODE: group retention by 'month' and take the mean of 'retention_pct'
avg_retention = retention.groupby(___)['retention_pct'].mean().round(1)

print('Average retention by month:')
print(avg_retention.to_string())

# Plot it
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(avg_retention.index, avg_retention.values, marker='o', linewidth=2,
        color='#3b82f6', markerfacecolor='white', markeredgewidth=2, markersize=7)
ax.fill_between(avg_retention.index, avg_retention.values, alpha=0.1, color='#3b82f6')

# Annotate each point
for x, y in zip(avg_retention.index, avg_retention.values):
    ax.annotate(f'{y:.0f}%', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=8, color='#6b7280')

ax.set_xlabel('Month Since First Payment', fontsize=10)
ax.set_ylabel('Avg Retention %', fontsize=10)
ax.set_title('Average Patron Retention Curve', fontsize=13, fontweight='bold')
ax.set_ylim(0, 110)
ax.set_xticks(avg_retention.index)
ax.grid(axis='y', alpha=0.3)
ax.set_facecolor('#f9fafb')
plt.tight_layout()
plt.savefig('outputs/avg_retention_curve.png', bbox_inches='tight')
plt.show()

---
## 10. Key insights — write them yourself

This is the most important cell in the notebook.
A chart without interpretation is just decoration.

**Your task:** Look at the heatmap and the retention curve and answer these
questions in plain English below. Write them as if you're presenting to
a non-technical stakeholder.

1. At what month does the biggest average drop-off happen?
2. Which cohort has the best long-term retention? Why might that be?
3. Which cohort shows the worst drop? What could explain it?
4. Are early cohorts (2024) better or worse than recent ones (2025–2026)?
5. What would you recommend to the creator based on this data?

In [ ]:
insights = """
YOUR INSIGHTS HERE — write in plain English, not code.

1. Biggest drop-off month:

2. Best cohort and possible reason:

3. Worst cohort and possible explanation:

4. Early vs recent cohort comparison:

5. Recommendation:
"""

print(insights)

---
## 11. Export the retention table

Save the long-form retention table as a CSV. This is what you'll
eventually load into the React dashboard instead of hardcoded data.

**Your task:** Run this, then open the CSV in Excel and verify it looks correct.

In [ ]:
retention.to_csv('data/processed/cohort_retention.csv', index=False)
print(f'Saved {len(retention)} rows to data/processed/cohort_retention.csv')
print()
print('Preview:')
print(retention.head(10).to_string(index=False))

---
## What's next?

Once you've completed this notebook and written your own insights in cell 10, the next steps are:

- **`02_churn_analysis.ipynb`** — cross the exit survey reasons with cohort data to explain *why* the drops happened
- **`03_tier_analysis.ipynb`** — does Full Rewards retain better than other tiers?
- **`04_mrr_timeline.ipynb`** — reconstruct monthly recurring revenue over time

Each notebook follows the same structure as this one:
load → filter → transform → visualize → write insights.

**Commit checklist before pushing to GitHub:**
- [ ] All cells run top-to-bottom without errors
- [ ] Cell 10 insights are filled in with your own words
- [ ] Charts saved to `outputs/`
- [ ] `patron_id_mapping.csv` is NOT in the commit
- [ ] Commit message is descriptive (e.g. `analysis: complete cohort retention with avg curve`)

---
*Analysis based on anonymized data from a content creator's Patreon page (2024–2026).*